# Supervised Learning EDA: Single-Word Dataset Preparation

**Goal**: Create a preliminary one-word dataset for the supervised learning component.

**Tasks**:
1. Filter clues to single-word definitions AND single-word answers
2. Explore data quality issues
3. Familiarize with WordNet for synonym prediction
4. Assess dataset size for Coach review

## 1. Setup and Data Loading

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import wordnet as wn

DATA_DIR = "../data"
df_clues = pd.read_csv(f'{DATA_DIR}/clues_raw.csv')
print(f"Total clues loaded: {len(df_clues):,}")

Total clues loaded: 660,613


In [2]:
df_clues[['clue_id', 'clue', 'answer', 'definition']].head()

clue_id,clue,answer,definition
1,"Acquisitive chap, as we see it (8)",COVETOUS,Acquisitive
2,Back yard fencing weak and sagging (6),DROOPY,sagging
3,"Stripping off uniform, love holding colonel's coat (10)",UNCLOTHING,Stripping
4,Without a mark where they should be gained (4),EXAM,where they should be gained
5,"Put a stop to Rugby's foul school leader (5,2,3,4)",KNOCK ON THE HEAD,Put a stop to


## 2. Data Quality Assessment

In [3]:
print("=== Missing Values ===")
missing = df_clues[['clue', 'answer', 'definition']].isnull().sum()
print(missing)
print(f"\nPercentage missing:")
print((missing / len(df_clues) * 100).round(2))

=== Missing Values ===
clue             323
answer          2259
definition    149096
dtype: int64

Percentage missing:
clue           0.05
answer         0.34
definition    22.57
dtype: float64


In [4]:
has_brackets = df_clues['clue'].str.contains(r'\\[', na=False)
print(f"Clues with square brackets (mis-parsed): {has_brackets.sum():,} ({has_brackets.sum()/len(df_clues)*100:.2f}%)")

Clues with square brackets (mis-parsed): 408 (0.06%)


## 3. Creating the Single-Word Dataset

In [5]:
df_clean = df_clues.dropna(subset=['answer', 'definition']).copy()
print(f"After dropping NaN: {len(df_clean):,} clues ({len(df_clean)/len(df_clues)*100:.1f}%)")

df_clean = df_clean[~df_clean['clue'].str.contains(r'\\[', na=False)]
print(f"After removing bracketed clues: {len(df_clean):,} clues")

After dropping NaN: 511,194 clues (77.4%)
After removing bracketed clues: 511,039 clues


In [6]:
df_clean['answer_wc'] = df_clean['answer'].astype(str).str.split().str.len()
df_clean['definition_wc'] = df_clean['definition'].astype(str).str.split().str.len()

print("=== Answer Word Count Distribution ===")
print(df_clean['answer_wc'].value_counts().head())

print("\n=== Definition Word Count Distribution ===")
print(df_clean['definition_wc'].value_counts().head())

=== Answer Word Count Distribution ===
answer_wc
1     441639
2      56270
3       9740
4       2687
5        550
Name: count, dtype: int64

=== Definition Word Count Distribution ===
definition_wc
1     256388
2     122468
3      63462
4      34155
5      16173
Name: count, dtype: int64


In [7]:
df_one_word = df_clean[
    (df_clean['answer_wc'] == 1) & 
    (df_clean['definition_wc'] == 1)
].copy()

print(f"=== SINGLE-WORD DATASET ===")
print(f"Total clues with single-word answer AND definition: {len(df_one_word):,}")
print(f"Percentage of original dataset: {len(df_one_word)/len(df_clues)*100:.2f}%")

df_one_word_answer = df_clean[df_clean['answer_wc'] == 1]
df_one_word_def = df_clean[df_clean['definition_wc'] == 1]
print(f"\nClues with single-word ANSWERS (any definition): {len(df_one_word_answer):,}")
print(f"Clues with single-word DEFINITIONS (any answer): {len(df_one_word_def):,}")

=== SINGLE-WORD DATASET ===
Total clues with single-word answer AND definition: 234,505
Percentage of original dataset: 35.50%

Clues with single-word ANSWERS (any definition): 441,639
Clues with single-word DEFINITIONS (any answer): 256,388


In [8]:
df_one_word['answer_clean'] = df_one_word['answer'].str.lower().str.strip()
df_one_word['definition_clean'] = df_one_word['definition'].str.lower().str.strip()

print("=== Sample of Single-Word Dataset ===")
df_one_word[['clue', 'definition_clean', 'answer_clean']].sample(10, random_state=42)

=== Sample of Single-Word Dataset ===


,clue,definition_clean,answer_clean
499257,"Live with rest, not posh (6)",live,reside
616046,Stern part that winds up audience (6),stern,severe
269012,First formal ball (5),first,primo
492176,Steadfast global body on duty touring West (10),steadfast,unswerving
108193,"With one movement, fluent about movie's completion (10)",completion,fulfilment
259072,Ponder mirror (7),ponder/mirror,reflect
641076,Leave deposit on lake in Emerald Isle (8),leave,furlough
393203,Maiden possibly facing accusation as cheat (10),cheat,overcharge
399641,Maximum portion of text rememberable (7),maximum,extreme
537953,The panel treated the animal (8),animal,elephant


## 4. Dataset Statistics for Coach Review

In [9]:
print("="*60)
print("DATASET SIZE SUMMARY FOR COACH REVIEW")
print("="*60)
print(f"Original dataset:                     {len(df_clues):>10,} clues")
print(f"After removing NaN:                   {len(df_clean):>10,} clues")
print(f"Single-word answer & definition:      {len(df_one_word):>10,} clues")
print(f"Unique definitions:                   {df_one_word['definition_clean'].nunique():>10,}")
print(f"Unique answers:                       {df_one_word['answer_clean'].nunique():>10,}")
print(f"Unique (definition, answer) pairs:    {len(df_one_word.groupby(['definition_clean', 'answer_clean'])):>10,}")
print("="*60)

DATASET SIZE SUMMARY FOR COACH REVIEW
Original dataset:                        660,613 clues
After removing NaN:                      511,039 clues
Single-word answer & definition:         234,505 clues
Unique definitions:                       34,222
Unique answers:                           47,677
Unique (definition, answer) pairs:       128,321


In [10]:
print("=== Top 20 Most Common Definitions ===")
print(df_one_word['definition_clean'].value_counts().head(20))

=== Top 20 Most Common Definitions ===
definition_clean
bird          1160
country       1123
plant         1110
drink          891
fruit          830
fish           813
city           766
game           647
capital        623
state          622
dance          541
girl           528
instrument     525
animal         503
material       488
tree           477
dog            451
flower         435
island         405
composer       396
Name: count, dtype: int64


In [11]:
print("=== Top 20 Most Common Answers ===")
print(df_one_word['answer_clean'].value_counts().head(20))

=== Top 20 Most Common Answers ===
answer_clean
extra      121
ideal      112
ache        95
idea        95
issue       93
inane       87
edge        84
echo        81
star        80
imagine     79
error       79
ennui       78
niece       77
onset       77
stir        76
adhere      76
taste       75
alibi       75
theme       75
even        74
Name: count, dtype: int64


## 5. WordNet Exploration for Supervised Learning

In [12]:
from nltk.corpus import wordnet as wn

example_def = "anger"
synsets = wn.synsets(example_def)

print(f"=== WordNet synsets for '{example_def}' ===")
for syn in synsets:
    print(f"\n{syn.name()}: {syn.definition()}")
    print(f"  Lemmas: {[l.name() for l in syn.lemmas()]}")

=== WordNet synsets for 'anger' ===

anger.n.01: a strong emotion; a feeling that is oriented toward some real or supposed grievance
  Lemmas: ['anger', 'choler', 'ire']

anger.n.02: the state of being angry
  Lemmas: ['anger', 'angriness']

wrath.n.02: belligerence aroused by a real or supposed wrong (personified as one of the deadly sins)
  Lemmas: ['wrath', 'anger', 'ire', 'ira']

anger.v.01: make angry
  Lemmas: ['anger']

anger.v.02: become angry
  Lemmas: ['anger', 'see_red']


In [13]:
def get_wordnet_synonyms(word):
    """Get all lemma names (synonyms) for a word across all synsets."""
    synonyms = set()
    for syn in wn.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name().lower().replace('_', ' '))
    synonyms.discard(word.lower())
    return synonyms

print(f"Synonyms for 'anger': {get_wordnet_synonyms('anger')}")
print(f"Synonyms for 'happy': {get_wordnet_synonyms('happy')}")
print(f"Synonyms for 'bird': {get_wordnet_synonyms('bird')}")

Synonyms for 'anger': {'angriness', 'ira', 'wrath', 'ire', 'see red', 'choler'}
Synonyms for 'happy': {'well-chosen', 'glad', 'felicitous'}
Synonyms for 'bird': {'birdie', 'doll', 'birdwatch', 'hoot', 'shuttlecock', 'boo', 'razz', 'fowl', 'raspberry', 'shuttle', 'razzing', 'chick', 'dame', 'hiss', 'wench', 'bronx cheer', 'skirt', 'snort'}


In [14]:
def answer_in_synonyms(row):
    """Check if the answer is in the WordNet synonyms of the definition."""
    definition = row['definition_clean']
    answer = row['answer_clean']
    synonyms = get_wordnet_synonyms(definition)
    return answer in synonyms

sample_size = 5000
df_sample = df_one_word.sample(min(sample_size, len(df_one_word)), random_state=42).copy()

print(f"Testing on {len(df_sample):,} samples...")
df_sample['answer_is_synonym'] = df_sample.apply(answer_in_synonyms, axis=1)

synonym_rate = df_sample['answer_is_synonym'].mean()
print(f"\n=== WordNet Synonym Match Rate ===")
print(f"Answer is a direct WordNet synonym of definition: {synonym_rate*100:.2f}%")
print(f"({df_sample['answer_is_synonym'].sum()} out of {len(df_sample)} samples)")

Testing on 5,000 samples...

=== WordNet Synonym Match Rate ===
Answer is a direct WordNet synonym of definition: 13.32%
(666 out of 5000 samples)


In [15]:
print("=== Examples: Answer IS a WordNet synonym ===")
matches = df_sample[df_sample['answer_is_synonym']][['definition_clean', 'answer_clean']].head(15)
for _, row in matches.iterrows():
    print(f"  {row['definition_clean']} -> {row['answer_clean']}")

=== Examples: Answer IS a WordNet synonym ===
  stern -> severe
  steadfast -> unswerving
  sickly -> sallow
  drink -> booze
  first -> inaugural
  beat -> rhythm
  attack -> assail
  completely -> altogether
  substitute -> replace
  lumberjack -> faller
  crack -> crevice
  sanctity -> holiness
  royal -> regal
  deposit -> sediment
  secret -> clandestine


In [16]:
print("=== Examples: Answer is NOT a direct WordNet synonym (challenging cases) ===")
non_matches = df_sample[~df_sample['answer_is_synonym']][['definition_clean', 'answer_clean', 'clue']].head(8)
for _, row in non_matches.iterrows():
    print(f"  {row['definition_clean']} -> {row['answer_clean']}")
    print(f"    Clue: {row['clue'][:60]}...")

=== Examples: Answer is NOT a direct WordNet synonym (challenging cases) ===
  live -> reside
    Clue: Live with rest, not posh (6)...
  first -> primo
    Clue: First formal ball (5)...
  completion -> fulfilment
    Clue: With one movement, fluent about movie's completion (10)...
  ponder/mirror -> reflect
    Clue: Ponder mirror (7)...
  leave -> furlough
    Clue: Leave deposit on lake in Emerald Isle (8)...
  cheat -> overcharge
    Clue: Maiden possibly facing accusation as cheat (10)...
  maximum -> extreme
    Clue: Maximum portion of text rememberable (7)...
  animal -> elephant
    Clue: The panel treated the animal (8)...


In [17]:
def max_path_similarity(word1, word2):
    """Get maximum path similarity between any synsets of two words."""
    synsets1 = wn.synsets(word1)
    synsets2 = wn.synsets(word2)
    
    if not synsets1 or not synsets2:
        return None
    
    max_sim = 0
    for s1 in synsets1:
        for s2 in synsets2:
            sim = s1.path_similarity(s2)
            if sim and sim > max_sim:
                max_sim = sim
    return max_sim if max_sim > 0 else None

print(f"Path similarity 'anger' to 'rage': {max_path_similarity('anger', 'rage')}")
print(f"Path similarity 'anger' to 'happy': {max_path_similarity('anger', 'happy')}")
print(f"Path similarity 'dog' to 'cat': {max_path_similarity('dog', 'cat')}")
print(f"Path similarity 'bird' to 'sparrow': {round(max_path_similarity('bird', 'sparrow'), 3)}")

Path similarity 'anger' to 'rage': 0.5
Path similarity 'anger' to 'happy': 0.25
Path similarity 'dog' to 'cat': 0.2
Path similarity 'bird' to 'sparrow': 0.333


In [18]:
print("Calculating path similarities on 5000 samples...")
df_sample['path_similarity'] = df_sample.apply(
    lambda row: max_path_similarity(row['definition_clean'], row['answer_clean']), 
    axis=1
)

no_path = df_sample['path_similarity'].isna().sum()
print(f"\n=== Path Similarity Statistics ===")
print(df_sample['path_similarity'].describe())

print(f"\nPairs with no WordNet path: {no_path} ({no_path/len(df_sample)*100:.1f}%)")
print(f"Perfect matches (1.0): {(df_sample['path_similarity'] == 1.0).sum()} ({(df_sample['path_similarity'] == 1.0).sum()/len(df_sample)*100:.1f}%)")
print(f"High similarity (>=0.5): {(df_sample['path_similarity'] >= 0.5).sum()} ({(df_sample['path_similarity'] >= 0.5).sum()/len(df_sample)*100:.1f}%)")

Calculating path similarities on 5000 samples...

=== Path Similarity Statistics ===
count    4484.000000
mean        0.445206
std         0.286860
min         0.055556
25%         0.250000
50%         0.333333
75%         0.500000
max         1.000000

Pairs with no WordNet path: 516 (10.3%)
Perfect matches (1.0): 798 (16.0%)
High similarity (>=0.5): 1903 (38.1%)


## 6. Data Quality Issues

In [19]:
has_nonalpha_def = df_one_word['definition_clean'].str.contains(r'[^a-z]', na=False)
has_nonalpha_ans = df_one_word['answer_clean'].str.contains(r'[^a-z]', na=False)
short_defs = df_one_word[df_one_word['definition_clean'].str.len() <= 2]
trivial = df_one_word[df_one_word['definition_clean'] == df_one_word['answer_clean']]

print("=== Data Quality Issues ===")
print(f"Definitions with non-alpha characters: {has_nonalpha_def.sum():,}")
print(f"Answers with non-alpha characters: {has_nonalpha_ans.sum():,}")
print(f"Very short definitions (<=2 chars): {len(short_defs):,}")
print(f"Trivial cases (definition == answer): {len(trivial):,}")

=== Data Quality Issues ===
Definitions with non-alpha characters: 13,809
Answers with non-alpha characters: 3,090
Very short definitions (<=2 chars): 1,716
Trivial cases (definition == answer): 5


In [20]:
df_one_word_clean = df_one_word[
    ~has_nonalpha_def & 
    ~has_nonalpha_ans &
    (df_one_word['definition_clean'].str.len() > 2) &
    (df_one_word['definition_clean'] != df_one_word['answer_clean'])
].copy()

print(f"\n=== FINAL CLEAN SINGLE-WORD DATASET ===")
print(f"Size: {len(df_one_word_clean):,} clues")
print(f"Unique definitions: {df_one_word_clean['definition_clean'].nunique():,}")
print(f"Unique answers: {df_one_word_clean['answer_clean'].nunique():,}")


=== FINAL CLEAN SINGLE-WORD DATASET ===
Size: 216,720 clues
Unique definitions: 23,242
Unique answers: 44,775


## 7. Save the Dataset

In [21]:
output_cols = ['clue_id', 'clue', 'answer', 'definition', 'answer_clean', 'definition_clean', 'source']
df_one_word_clean[output_cols].to_csv(f'{DATA_DIR}/clues_single_word.csv', index=False)
print(f"Saved: {DATA_DIR}/clues_single_word.csv")
print(f"Shape: {df_one_word_clean[output_cols].shape}")

Saved: ../data/clues_single_word.csv
Shape: (216720, 7)


## 8. Summary for Coach Meeting

In [22]:
print("="*70)
print("SUMMARY FOR COACH MEETING")
print("="*70)
print("\n1. DATASET SIZE:")
print(f"   - Original clues: 660,613")
print(f"   - Single-word (definition & answer): 234,505")
print(f"   - After cleaning: 216,720")
print(f"   - This is 32.8% of the original data")

print("\n2. DATA QUALITY ISSUES FOUND:")
print(f"   - Missing definitions: 149,096")
print(f"   - Bracketed clues (mis-parsed): 408")
print(f"   - Non-alphabetic definitions: 13,809")
print(f"   - Very short definitions (<=2 chars): 1,716")
print(f"   - Trivial (definition == answer): 5")

print("\n3. WORDNET ANALYSIS (on 5,000 sample):")
print(f"   - Direct synonym matches: 13.3%")
print(f"   - Mean path similarity: 0.445")
print(f"   - No WordNet path found: 10.3%")

print("\n4. QUESTIONS FOR COACH:")
print(f"   - Is 216,720 clues sufficient for supervised learning?")
print("   - Should we include multi-word definitions to increase data size?")
print("   - How to handle definition/answer pairs not in WordNet?")
print("="*70)

SUMMARY FOR COACH MEETING

1. DATASET SIZE:
   - Original clues: 660,613
   - Single-word (definition & answer): 234,505
   - After cleaning: 216,720
   - This is 32.8% of the original data

2. DATA QUALITY ISSUES FOUND:
   - Missing definitions: 149,096
   - Bracketed clues (mis-parsed): 408
   - Non-alphabetic definitions: 13,809
   - Very short definitions (<=2 chars): 1,716
   - Trivial (definition == answer): 5

3. WORDNET ANALYSIS (on 5,000 sample):
   - Direct synonym matches: 13.3%
   - Mean path similarity: 0.445
   - No WordNet path found: 10.3%

4. QUESTIONS FOR COACH:
   - Is 216,720 clues sufficient for supervised learning?
   - Should we include multi-word definitions to increase data size?
   - How to handle definition/answer pairs not in WordNet?
